<a href="https://colab.research.google.com/github/kgeorge-fission/Resources/blob/feat-prompt-evaluation/examples/flotorch-evaluation-notebooks/prompt-evaluation/prompt_evaluation.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Prompt Evaluation Notebook

This notebook evaluates multiple **system-user prompt and model pairs** for **question answering tasks** using the **Flotorch SDK** and **Flotorch Eval**. This uses context provided through either ground truth file or fetches context from the provided knowledge base.

## What it Does

This notebook runs structured benchmarking and prompt experimentation to identify the most effective prompting strategies for your LLM-based RAG or QA system. It supports the following capabilities:

### Core Evaluation Features
- Performs automated evaluation on LLM responses using `retrieved context`.
- Computes the following quality metrics:
  - **Context Precision** - Measures how focused the retrieved context is to the query.
  - **Faithfulness** - Measures factual consistency of the answer with the given context.
  - **Answer Relevancy** - Measures how well the answer addresses the question.
  - **Context Relevancy** - Evaluates the relevance of retrieved context to the question.
  - **Context Recall** - Measures how much of the necessary ground-truth information is present in the context.
  - **Hallucination** - Detects whether the answer contains unsupported or incorrect claims.
  - **Maliciousness** - Detects if the response contains harmful or unsafe content.

### Performance, Cost, and Ranking
- Calculates:
  - **Average metric score** per prompt pair
  - **Average response latency**
  - **Total tokens used**
  - **Total cost (USD)**
- Selects the **best-performing prompt pair** using a **weight-based scoring system**:
  - Assign weights to **Quality Score**, **Cost**, and **Latency**
  - The notebook identifies the best prompt pair based on your performance priorities

### Additional Advanced Capabilities
This notebook also supports several prompt engineering and experimentation features:

| Feature | Description |
|--------|-------------|
| **Models and Knowledge Base Validation** | Validates availability and configuration of all LLM, embedding models, and knowledge bases before running experiments |
| **Prompt Assembly** | Allows customizable LLM payload construction using JSON to define message structure (system, assistant, user roles, etc.) |

## Output

The notebook outputs:
- A list of evaluated prompt pairs with full metric breakdown
- Aggregate latency, cost, and token usage
- Rank-ordered list of prompt pairs based on your selected weight configuration
- The **top-performing prompt pair** and its details

## Requirements to Run the Notebook

You will need:
- A **Flotorch account** with LLM and embedding models configured
- A **`gt.json`** file with ground truth QA pairs (format shown later in the notebook)
- A **`prompts.json`** file containing system-user prompt pairs (format shown later)
- (Optional) An LLM configured for automatic prompt generation and question rephrasing

---

You can now run structured evaluations, cost-latency-quality tradeoff comparisons, automatic prompt engineering, and context experimentation all from one unified workflow.

![Diagram](/home/kiran/Flotorch-fork/Resources/examples/flotorch-evaluation-notebooks/prompt-evaluation/images/notebook-2.drawio.png)


In [ ]:
# Install flotorch-sdk and flotorch-core
# You can safely ignore the dependency errors during the installation.

!pip install flotorch==2.2.0b1 flotorch-eval==1.2.0b1 -q

### Model Setup  

Before running the experiment, you first need to configure your Flotorch environment so the notebook can communicate with your console and access the models you've created.  

Set the following variables:  
- **`FLOTORCH_API_KEY`** - your Flotorch API key.  
- **`FLOTORCH_BASE_URL`** - the base URL of your Flotorch console instance.  

---

Before running the experiment, define the core models that will be used throughout this notebook.  

#### 1. **`inferencer_model_names`**  
This is a list of **generative LLM model** used to produce answers for each question in the ground truth (GT) data.  
You can configure these model in the **Flotorch Console** by first adding a provider and then creating a model under that provider. Each model will be run across all the system-user prompt pairs and questions.

#### 2. **`evaluation_llm_model_name`**  
This is the **evaluation LLM model** used to assess the quality of the responses generated by the inferencer model.  
Similar to the inferencer model, it should also be created in the Flotorch Console under a provider of your choice.  

#### 3. **`evaluation_embedding_model_name`**  
This is the **embedding model** used during evaluation to measure semantic similarity between responses and reference answers.  
To set this up in the console, create a provider and append the model name to it.  

**Example:**  
If you create a provider named `openai-provider` and want to use the `text-embedding-ada-002` embedding model, then your `evaluation_embedding_model_name` will be:  

openai-provider/text-embedding-ada-002

#### **Best Practice:**
It’s generally recommended to keep your inferencer and evaluation models separate.
Using different models helps ensure that the evaluation remains unbiased — if the same model that generated the responses also judges them, it may overestimate its own performance. Separate models provide a more objective and reliable assessment.

In [ ]:
import getpass

try:
  FLOTORCH_API_KEY=getpass.getpass("Paste your API key here: ")
  print(f"Success")
except getpass.GetPassWarning as e:
    print(f"Warning: {e}")

FLOTORCH_BASE_URL=input("Paste your Flotorch Base URL here: ")

### Setup models, knowledge bases and other variables
Flotorch includes global provider LLMs that can be used immediately with no additional setup, or you can define your own custom models in the console.

This section also defines shared configuration variables that control important parameters used throughout the notebook.


In [ ]:
# ============================================================================
# Global Provider Models (Flotorch Gateway Models)
# ============================================================================
# These models are available from the Flotorch gateway and can be used
# for evaluation, agent operations, and other tasks.
# You can set up your own models in the Flotorch Console and use them here.

MODEL_CLAUDE_HAIKU = "flotorch/flotorch-claude-haiku-4-5"
MODEL_CLAUDE_SONNET = "flotorch/flotorch-claude-sonnet-3-5-v2"
MODEL_CLAUDE_4_5_SONNET = "flotorch/flotorch-claude-sonnet-4-5"
MODEL_AWS_NOVA_PRO = "flotorch/flotorch-aws-nova-pro"
MODEL_AWS_NOVA_LITE = "flotorch/flotorch-aws-nova-lite"
MODEL_AWS_NOVA_MICRO = "flotorch/flotorch-aws-nova-micro"


# Variables to set

# Flotorch LLM model for inferencing
inference_model_names = [MODEL_AWS_NOVA_LITE]

# Flotorch LLM model for evaluation
evaluation_llm_model_name = MODEL_CLAUDE_HAIKU

# Flotorch embedding model for evaluation
# use the format <model_provider>/<model_name> where <model_provider> is the provider you've set up in Flotorch Console and <model_name> is the actual model name.
# Eg: openai-provider/text-embedding-ada-002
evaluation_embedding_model_name = "openai-provider/text-embedding-ada-002"

# This is the knowledge base repository you have set up on Flotorch console.
# This is optional and can be left as an empty string or commented out if you don't have a knowledge base.
# Note: Some metrics that depend on the cotext like 'context relevancy' will not be used if knowledge base is not passed.
knowledge_base_repo = "bedrock-kb"

### Load Project Code

This step retrieves the project source code from the repository and makes the `prompt_evaluation` module available for import.  
All core components—including prompt generation, evaluation logic, scoring utilities, and workflow orchestration—are defined in this module, so it must be loaded before running the evaluation pipeline.


In [ ]:
!git clone --branch feat-prompt-evaluation https://github.com/kgeorge-fission/Resources.git
import sys
sys.path.append("/content/Resources/examples/flotorch-evaluation-notebooks/prompt-evaluation")

### Imports  

The following libraries and modules are required for this notebook.  
They include flotorch eval imports and other core modules with utilities designed for this notebook.


In [ ]:
#Required imports
import json

from flotorch_eval.llm_eval import LLMEvaluator, MetricKey

from utils import (
    ExperimentRunner,
    validate_environment,
    EvaluationDatasetType,
    PromptEvaluationResult,
    display_prompt_results,
    best_prompt_pair,
    get_weighted_scores
)


### Upload Ground Truth and Prompts Files  

Use the file upload widgets below to upload your **Ground Truth (gt.json)** and **Prompts (prompts.json)** files.  
The notebook will automatically read and load them into their variables which will be used in later cells.

Context can be provided either through the ground truth or as a Flotorch knowledge base.

#### **Expected File Formats**

**Ground Truth (`gt.json`)**
```json
[
  {
    "question": "What is Amazon Bedrock?",
    "answer": "Amazon Bedrock is a fully managed service that makes foundation models available through an API.",
    "context": [<item1>, <item2>...]
  },
    {
    "question": "Which FMs are available on Amazon Bedrock?",
    "answer": "Amazon Bedrock customers can choose from some of the most cutting-edge FMs available today. Currently we offer 47 models.",
    "context": [<item1>, <item2>...]
  },
  ...
]
```

**Prompt pairs (`prompts.json`)**
```json
[
  {
    "system_prompt": "You are an AI assistant that provides accurate answers based on the given context.",
    "user_prompt": "Answer the following question using the provided context."
  },
  {
    "system_prompt": "You are an AWS cloud documentation assistant trained on Amazon Bedrock materials. Read the provided context carefully and answer only from it.",
    "user_prompt": "Using only the retrieved Bedrock context, give a short and factual answer to the question below."
  },
  ...
]
```

You can either:
- Upload your files to your Google drive and load the files here by providing the path: This is better if you have multiple files and plan to switch them frequently.

OR

- You can upload the files directly in the cell below and those will be stored in the Google Colab temporary storage for the experimentation: This is good if you only plan to run a file once.

In [ ]:
# If you don't have files in your local directory, you can use the following code to upload them.

# Uncomment the below code if you're planning to do this

from google.colab import files
print("Please upload your Ground Truth file (gt.json)")
gt_upload = files.upload()

gt_path = list(gt_upload.keys())[0]
with open(gt_path, 'r') as f:
    ground_truth = json.load(f)
print(f"Ground truth loaded successfully — {len(ground_truth)} items\n")


print("Please upload your Prompts file (prompts.json)")
prompts_upload = files.upload()

prompts_path = list(prompts_upload.keys())[0]
with open(prompts_path, 'r') as f:
    prompts = json.load(f)
print(f"Prompts loaded successfully — {len(prompts)} prompt pairs")

In [ ]:
# If you already have files in your local directory, you can use the following code to load them.

# Uncomment the below code if you're planning to do this

# GT_PATH = "data/gt_test.json"
# with open(GT_PATH, 'r') as f:
#     ground_truth = json.load(f)
# print(f"Ground truth loaded successfully — {len(ground_truth)} items\n")

# PROMPT_PATH = "data/prompts_test.json"
# with open(PROMPT_PATH, 'r') as f:
#     prompts = json.load(f)
# print(f"Prompt loaded successfully.")

## Run Experiments  

This section defines and executes the asynchronous experiment workflow.  

It initializes the asynchronous functions responsible for running experiments and performing knowledge base searches. The process iterates through each **model** in the list, through each **system-user prompt pair**, runs it against all question-answer entries in the ground truth dataset, and compiles the results into a structured dictionary. This continues until every prompt pair has been evaluated.  

**Note:** This is a compute-intensive process since each model is executed for every question and prompt pair.  
For example, with **3 models**, **10 prompt pairs** and **10 questions**, the total number of LLM queries will be **3 x 10 x 10 = 300**.


### Assembly Rule

The **assembly rule** defines how different components — such as the system prompt, user prompt, context, n-shot examples, and questions — are combined to form the final input for the model. The way these pieces are stitched together significantly impacts the quality of the model’s responses.

This notebook allows users to **easily customize** the assembly structure through a JSON object.

#### **Default Assembly Structure**
```python
assembly_rule = {
    "separator": "",
    "system_prompt": ["system", "context", "examples"],
    "user_prompt": ["user", "question"]
}
```
#### How It Works

- Everything listed under system_prompt is concatenated (using the defined separator) and passed to the model as the system role.

- Everything listed under user_prompt is concatenated (using the same separator) and passed to the model as the user role.

Available Components

You can use the following values within either system_prompt or user_prompt:

`system` → The system prompt provided by the user.

`user` → The user prompt provided by the user.

`context` → The retrieved or provided contextual information.

`examples` → The n-shot examples included alongside the prompts.

`question` → Each question present in the ground truth (GT) data.

`Separator`

The "separator" key defines what string (e.g., "\n\n", "---", " ") is used to join multiple components.

You can set it to any value depending on how you want the content to be structured in the final payload.

This flexible rule-based approach makes it easy to experiment with different prompt assemblies and evaluate their impact on model performance.

In [ ]:
# Define how prompt components are assembled into the final LLM input
# This controls the structure of messages sent to the model
assembly_rule={
        "separator": "\n",  # String used to join multiple components
        "system_prompt": ["system"],  # Components that go into the system role
        "user_prompt": ["user", "context", "examples", "question"]  # Components that go into the user role
    }



#### **Configuration**

The `ExperimentRunner` class accepts the following parameters:

**Required Parameters:**
- **`models`** (`List[str]`) — List of LLM model names to test. Each model will be evaluated across all prompt pairs and questions. Models should be configured in your Flotorch Console (e.g., `["flotorch/haiku-long"]`).
- **`prompts`** (`List[Dict[str, Any]]`) — List of prompt dictionaries, each containing `system_prompt` and `user_prompt` keys. Optionally can include `examples` for n-shot learning.
- **`ground_truth`** (`List[Dict[str, Any]]`) — List of question-answer pairs. Each dictionary should contain `question` and `answer` keys. Optionally can include `context` for pre-provided context.
- **`api_key`** (`str`) — Your Flotorch API key for authentication.
- **`base_url`** (`str`) — The base URL of your Flotorch console instance.

**Optional Parameters:**
- **`knowledge_base`** (`Optional[str]`) — Name of the knowledge base repository configured in Flotorch Console. If provided, context will be retrieved from this knowledge base. If `None`, context must be provided in the ground truth data. Default: `None`.
- **`context_sizes`** (`Optional[List[int]]`) — List of context chunk counts to test (e.g., `[1, 3, 5]`). Each size will be tested for all model-prompt-question combinations. If `None`, all available context chunks will be used. Default: `None`.
- **`context_strategy`** (`str`) — Strategy for selecting context chunks when `context_sizes` is specified:
  - `"top"` — Uses the most relevant chunks based on similarity scores (default)
  - `"random"` — Randomly selects chunks (useful for testing robustness)
  Default: `"top"`.
- **`random_seed`** (`Optional[int]`) — Seed for reproducible random selection. Used when `context_strategy="random"` or when sampling n-shot examples. If set, ensures reproducible results across runs. Default: `None`.
- **`assembly_rule`** (`Optional[Dict]`) — Custom rule for assembling prompt components. Defines how system prompt, user prompt, context, examples, and question are combined. See the "Assembly Rule" section above for details. Default: `None` (uses default assembly).
- **`n`** (`Optional[int]`) — Number of n-shot examples to use from each prompt's examples list. If specified, randomly samples `n` examples from available examples. If fewer than `n` examples are available, uses all available examples and logs a warning. If `None`, uses all available examples. Default: `None`.

**Example:**
```python
runner = ExperimentRunner(
    models=["flotorch/haiku-long"],
    prompts=prompts,
    ground_truth=ground_truth,
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_BASE_URL,
    knowledge_base="bedrock-kb",
    context_sizes=[1, 3],
    context_strategy="top",
    random_seed=42,
    assembly_rule=assembly_rule,
    n=3  # Use 3 randomly sampled examples per prompt
)
```

The evaluation results will include separate metrics for each context size, allowing you to identify the optimal configuration for your use case.

In [ ]:
# Initialize the ExperimentRunner with your configuration
# This will run experiments for all combinations of models × prompts × questions
runner = ExperimentRunner(
    models=inference_model_names,  # List of LLM models to test
    prompts=prompts,  # List of prompt pairs (system + user prompts)
    ground_truth=ground_truth,  # List of question-answer pairs to evaluate
    api_key=FLOTORCH_API_KEY,  # Your Flotorch API key
    base_url=FLOTORCH_BASE_URL,  # Your Flotorch base URL
    assembly_rule=assembly_rule,  # How to combine prompt components
    n=1  # Number of n-shot examples to use (None = use all available)
)

# Run experiments asynchronously with concurrency control
# concurrency=10 means up to 10 experiments run in parallel
# This returns a dictionary with 'units_count', 'warnings', and 'runs'
evaluation_data = await runner.run_async(concurrency=10)

In [ ]:
evaluation_data

## Evaluate Results  

This section initializes the models required for evaluation using the model names you have provided and computes performance metrics for each system-user prompt pair.

The `run_evaluation()` function:  
- Initializes Flotorch LLMEvaluator using your inference llm and embedding model
- By default LLMEvaluator runs on all the metrics that is available.
Available metrics are:
    - Context precision
    - Context recall
    - Context relevancy
    - Answer relevancy
    - Hallucination
    - Faithfulness
    - Aspect critic
- Aspect critic requires a definition of the aspect and that is provided to the LLMEvaluator in the metric config.
- Providing the LLMEvaluator with headers activates gateway metrics:
    - Total tokens
    - Total cost
    - average and total latency
- Computes an overall **average score** for each prompt pair and returns a structured summary of all evaluations.

### **What does these metrics mean?**

**Context Precision:**  
Measures how well the retrieved context supports the generated answer. High context precision indicates that the model effectively retrieves focused and useful information rather than unrelated or noisy context.

**Context recall**
Measures how much of the relevant information from the source context was used in the generated answer. High context recall means the model successfully captured most of the necessary details from the retrieved context.

**Context Relevancy**
Evaluates how relevant the retrieved context is to the user's question. High context relevancy indicates that the retrieved passages are strongly aligned with what the question is asking.

**Answer Relevancy:**  
Evaluates how relevant and appropriate the model's answer is to the user's question.

**Hallucination**
Assesses whether the model introduces information not present or supported by the provided context. A lower hallucination score indicates that the model stays faithful to the retrieved data without adding fabricated or unrelated details.

**Faithfulness:**  
Assesses whether the generated answer is factually consistent with the retrieved context.

**Maliciousness:**  
An aspect-critic metric that determines whether the model's response contains harmful, toxic, or unsafe content. It assesses the presence of language or reasoning that could be considered offensive, dangerous, or unethical in nature.

*Note:* You may see some warnings or errors printed to the console during execution. These come from the RAGAS library and can be safely ignored — they do not affect the evaluation flow.


In [ ]:
def run_evaluation(evaluation_data: EvaluationDatasetType) -> PromptEvaluationResult:
    """
    Evaluates experiment results using Flotorch Eval.
    
    This function:
    1. Validates that evaluation models are available
    2. Initializes the LLMEvaluator with specified metrics
    3. Evaluates each prompt set from the experiment data
    4. Computes average scores and aggregates gateway metrics (cost, latency, tokens)
    5. Returns structured results for each prompt pair
    
    Args:
        evaluation_data: Dictionary containing experiment results with 'runs' key
        
    Returns:
        List of dictionaries, each containing evaluation metrics for a prompt pair
    """
    import math

    # Extract the runs list from evaluation data
    # Falls back to evaluation_data itself if 'runs' key doesn't exist
    data_runs = evaluation_data.get("runs", evaluation_data)

    # Configure aspect-critic metrics (custom evaluation criteria)
    # Here we define 'maliciousness' as a custom metric
    metric_args = {
        MetricKey.ASPECT_CRITIC: {
            "maliciousness": {
                "name": "maliciousness",
                "definition": "Is the response harmful or toxic?"
            }
        }
    }

    # Validate that the evaluation models are configured and accessible
    validate_environment(
        llm_models=[evaluation_llm_model_name],
        embedding_models=[evaluation_embedding_model_name],
        api_key=FLOTORCH_API_KEY,
        base_url=FLOTORCH_BASE_URL
    )

    # Initialize the LLM evaluator client
    # This will be used to compute quality metrics for each prompt pair
    metrics_to_compute = [
        MetricKey.ANSWER_RELEVANCE,
        MetricKey.ASPECT_CRITIC,
        MetricKey.CONTEXT_PRECISION,
        MetricKey.CONTEXT_RECALL,
        MetricKey.CONTEXT_RELEVANCY,
        MetricKey.HALLUCINATION,
        MetricKey.FAITHFULNESS
        ]
    client = LLMEvaluator(
        api_key=FLOTORCH_API_KEY,
        base_url=FLOTORCH_BASE_URL,
        embedding_model=evaluation_embedding_model_name,  # For semantic similarity calculations
        inferencer_model=evaluation_llm_model_name,  # For LLM-based metric evaluation
        metrics=metrics_to_compute,  # Metrics to compute
        evaluation_engine='auto',  # Evaluation framework to use
        metric_configs=metric_args  # Custom metric configurations
    )

    results = []

    # Evaluate each prompt set from the experiment data
    for prompt_set in data_runs:
        try:
            # Run evaluation on all experiments for this prompt pair
            eval_result = client.evaluate(prompt_set.get("experiments"))
            
            # Extract evaluation metrics (quality scores) and gateway metrics (cost, latency, tokens)
            eval_metrics = eval_result.get("evaluation_metrics", {})
            gateway_metrics = eval_result.get("gateway_metrics", {})

            # Filter out non-numeric and invalid (NaN/Inf) values to compute average
            numeric_values = [
                v for v in eval_metrics.values()
                if isinstance(v, (int, float))
                and not (isinstance(v, float) and (math.isnan(v) or math.isinf(v)))
            ]

            # Compute average score across all metrics for this prompt pair
            if numeric_values:
                average_score = sum(numeric_values) / len(numeric_values)
                eval_metrics['average_score'] = round(average_score, 2)
            else:
                eval_metrics['average_score'] = 0.0

            # Merge gateway metrics (cost, latency, tokens) into evaluation metrics
            if gateway_metrics:
                eval_metrics.update(gateway_metrics)

            # Store results for this prompt pair
            results.append(
                {
                    "inference_model": prompt_set.get("model"),
                    "system_prompt": prompt_set.get("system_prompt"),
                    "user_prompt": prompt_set.get("user_prompt"),
                    "context_size": prompt_set.get("context_size"),
                    "evaluation_metrics": eval_metrics
                }
            )

        except Exception as e:
            # Handle errors gracefully - log and continue with other prompt pairs
            print(f"Error evaluating prompt set: {e}")
            results.append(
                {
                    "inference_model": prompt_set.get("model"),
                    "system_prompt": prompt_set.get("system_prompt"),
                    "user_prompt": prompt_set.get("user_prompt"),
                    "context_size": prompt_set.get("context_size"),
                    "evaluation_metrics": {
                        "average_score": 0.0,
                        "error": str(e)
                    }
                }
            )

    return results


In [ ]:
# Run evaluation on the experiment data
# This computes quality metrics (answer relevancy, maliciousness, etc.) for each prompt pair
# and aggregates cost, latency, and token usage information
results: PromptEvaluationResult = run_evaluation(evaluation_data)

In [ ]:
# Display the evaluation results
# This shows the computed metrics (answer relevancy, maliciousness, etc.) for each prompt pair
# along with cost, latency, and token usage information
results

### Weighted Scoring Configuration

This section allows you to **assign weights** to different evaluation dimensions —  
**response quality**, **latency**, and **cost** — based on what matters most for your use case.

The notebook will then compute a new **`weighted_final_score`** for each prompt-model pair using these weights.

---

#### **How Weighted Scoring Works**

Each evaluated prompt pair produces:
- `average_score` → Quality of the answer (**higher is better**)  
- `average_latency_ms` → Speed of generation (**lower is better**)  
- `average_cost` → Cost in USD (**lower is better**)

Since latency and cost are *better when lower*, they are automatically **normalized and inverted** during weighting so that all values align directionally (**higher = better**).

---

#### **Set Your Custom Weights**

Adjust the following values based on your priorities:  

| Metric | Description | Suggested Range | Example |
|:--|:--|:--:|:--:|
| `average_score` | Importance of response quality | 0.0-1.0 | `0.6` |
| `average_latency_ms` | Importance of response speed | 0.0-1.0 | `0.2` |
| `average_cost` | Importance of cost efficiency | 0.0-1.0 | `0.2` |

**Note:**  
- The notebook will automatically normalize your weights so they sum to 1.  
- If you skip this section, default weights of **(0.6, 0.2, 0.2)** will be used.

---


In [ ]:
# Define weights for weighted scoring
# These weights determine how much importance to give to quality vs speed vs cost
# The weights will be automatically normalized to sum to 1.0
user_weights = {
    "average_score": 0.4,  # Weight for response quality (higher = better)
    "average_latency_ms": 0.3,  # Weight for response speed (lower = better, will be inverted)
    "average_cost": 0.3  # Weight for cost efficiency (lower = better, will be inverted)
}

# Apply weighted scoring to all results
# This computes a weighted_final_score for each prompt pair based on your priorities
results = get_weighted_scores(results, user_weights)

In [ ]:
# Display the best performing prompt pair based on weighted scores
# This shows the prompt pair with the highest weighted_final_score
best_prompt_pair(results)

In [ ]:
# Display the evaluation results after applying weighted scoring
# This shows the updated results with weighted_final_score for each prompt pair
results

In [ ]:
# Display a comprehensive table of all evaluation results
# Shows metrics, scores, costs, and latencies for all prompt pairs
# Results are sorted by weighted score (if available) or average score
display_prompt_results(results)